In [41]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata
import urllib.request
import urllib
import pandas as pd
import sys


# # 검색 키워드 URL 변환
# import urllib.parse

# # 날짜 및 시간
# from datetime import datetime
# Step 2. 사용자에게 수집 조건을 입력받습니다.

print("=" * 80)
print("유튜브 숏츠 댓글 수집 프로그램")
print("=" * 80)


# collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

# while collect_type not in ['1', '2']:
#     print("입력 오류입니다. 1 또는 2만 입력하세요.")
#     collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

# # 기본 URL
# default_url = "https://www.youtube.com/shorts/UkU_P-jfBtM"

# # URL 또는 키워드 입력
# if collect_type == '1':
#     target_url = input("\n댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()

#     if target_url == "":
#         target_url = default_url

#     search_keyword = ""

# else:
#     search_keyword = input("\n검색할 키워드를 입력하세요: ").strip()
#     target_url = ""

# 1. 수집 방식 선택
print("\n[수집 방식 선택]")
print("1. URL을 직접 입력하여 댓글 수집")
print("2. 키워드로 영상 검색 후 댓글 수집")

collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

while collect_type not in ['1', '2']:
    print("입력 오류입니다. 1 또는 2만 입력하세요.")
    collect_type = input("원하는 방식을 선택하세요 (1 또는 2): ").strip()

#"https://www.youtube.com/shorts/UkU_P-jfBtM"
# 2. URL 또는 키워드 입력
if collect_type == '1':
    target_url = input("\n댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()
    search_keyword = ""
else:
    search_keyword = input("\n검색할 키워드를 입력하세요: ").strip()
    target_url = ""


# 3. 수집할 댓글 수 입력
try:
    comment_cnt = int(input("\n수집할 댓글 수를 입력하세요(기본값: 30): ").strip())
except ValueError:
    comment_cnt = 30
    print("숫자가 입력되지 않아 기본값 30건으로 설정합니다.")

if comment_cnt <= 0:
    comment_cnt = 30
    print("0 이하의 값은 입력할 수 없어 기본값 30건으로 설정합니다.")

# 4. 저장 폴더명 입력
save_folder = input("\n저장할 폴더명을 입력하세요(기본값:c:\\py_temp\\): ").strip()

if save_folder == "":
    save_folder = "c:\\py_temp\\"

# 5. 입력 결과 확인
print("\n" + "=" * 80)
print("입력한 수집 조건")
print("=" * 80)

if collect_type == '1':
    print("수집 방식   : URL 직접 입력")
    print("대상 URL    :", target_url)
else:
    print("수집 방식   : 키워드 검색")
    print("검색 키워드 :", search_keyword)

print("수집 댓글 수 :", comment_cnt)
print("저장 폴더명  :", save_folder)
print("=" * 80)
# # 더 안전하게 쓰는 코드 
# # 2. URL 또는 키워드 입력
# if collect_type == '1':
#     target_url = input("\n댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()
    
#     while target_url == "":
#         print("URL이 비어 있습니다. 다시 입력하세요.")
#         target_url = input("댓글을 수집할 유튜브 숏츠 URL을 입력하세요: ").strip()

#     if not target_url.startswith("http"):
#         print("URL 앞에 https:// 가 없어 자동으로 붙입니다.")
#         target_url = "https://" + target_url

#     search_keyword = ""

# else:
#     search_keyword = input("\n검색할 키워드를 입력하세요: ").strip()
#     target_url = ""
#Step 3. 크롬 드라이버 설정 및 웹 페이지 열기 ~ 일단 유튜브에 들어가자 
s = Service("c:/py_temp/chromedriver.exe") 
driver = webdriver.Chrome(service=s) 

url = 'https://www.youtube.com' 
driver.get(url) 
driver.maximize_window() 
time.sleep(3) 


# Step 4. 선택한 방식에 따라 실제 페이지로 이동하기

if collect_type == '1':
    # URL 직접 입력 방식
    driver.get(target_url)
    time.sleep(3)
    print("\n입력한 숏츠 URL로 이동했습니다.")

elif collect_type == '2':
    # 키워드 검색 방식
    search_box = driver.find_element(By.NAME, "search_query")
    search_box.clear()
    search_box.send_keys(search_keyword)
    search_box.send_keys(Keys.ENTER)
    time.sleep(3)
    print("\n키워드 검색 결과 페이지로 이동했습니다.")
print("collect_type :", collect_type)
print("target_url   :", target_url)
print("search_keyword :", search_keyword)
# Step 5. 동영상 제목 가져오기

try:
    video_title = driver.find_element(
        By.CSS_SELECTOR,
        "yt-shorts-video-title-view-model h2 span"
    ).text

    video_title = video_title.split("#")[0].strip()

    print("동영상 제목 :", video_title)

except:
    video_title = "제목 없음"
    print("동영상 제목을 가져오지 못했습니다.")

video_url = driver.current_url

video_no = 1

comment_btn = driver.find_element(
    By.CSS_SELECTOR,
    'button[aria-label*="댓글"]'
)

comment_btn.click()
time.sleep(5)



유튜브 숏츠 댓글 수집 프로그램

[수집 방식 선택]
1. URL을 직접 입력하여 댓글 수집
2. 키워드로 영상 검색 후 댓글 수집



입력한 수집 조건
수집 방식   : URL 직접 입력
대상 URL    : https://www.youtube.com/shorts/EpRSKZ_S9us
수집 댓글 수 : 50
저장 폴더명  : c:\py_temp\

입력한 숏츠 URL로 이동했습니다.
collect_type : 1
target_url   : https://www.youtube.com/shorts/EpRSKZ_S9us
search_keyword : 
동영상 제목 : 유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청


In [49]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# import time

# # ---------------------------------
# # 1. 댓글 목록 영역 찾기
# # ---------------------------------
# comment_list = driver.find_element(
#     By.CSS_SELECTOR,
#     'ytd-comments[engagement-panel] ytd-item-section-renderer #contents'
# )

# # 댓글창에 포커스 주기
# ActionChains(driver).move_to_element(comment_list).click().perform()
# time.sleep(2)

# # ---------------------------------
# # 2. 원하는 댓글 수까지 스크롤
# # ---------------------------------
# target_count = comment_cnt   # 사용자가 입력한 목표 댓글 수
# last_count = 0
# same_count = 0

# while True:
#     comments = driver.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer')
#     current_count = len(comments)
#     print(f"현재 로딩된 댓글 수 : {current_count}")

#     # 목표 개수 도달하면 종료
#     if current_count >= target_count:
#         print("원하는 댓글 수만큼 로드 완료")
#         break

#     # 댓글창 내부를 아래로 스크롤
#     driver.execute_script("arguments[0].scrollTop += 800;", comment_list)
#     time.sleep(2)

#     # 새 댓글 수 다시 확인
#     comments = driver.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer')
#     new_count = len(comments)

#     # 더 이상 늘어나지 않으면 카운트
#     if new_count == current_count:
#         same_count += 1
#     else:
#         same_count = 0

#     # 여러 번 반복해도 안 늘어나면 종료
#     if same_count >= 5:
#         print("더 이상 새로운 댓글이 로딩되지 않습니다.")
#         break

In [ ]:
# by jin 
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.common.action_chains import ActionChains

# for a in range(1, page_cnt + 1):
#     try:
#         # 1. 댓글창 컨테이너를 정확히 타겟팅
#         scroll_box = driver.find_element(By.CSS_SELECTOR, "ytd-engagement-panel-section-list-renderer #content")
        
#         # 2. 현재 로드된 댓글 중 가장 마지막 댓글 요소를 찾음
#         comments = driver.find_elements(By.CSS_SELECTOR, "ytd-comment-thread-renderer")
#         if comments:
#             last_comment = comments[-1]
#             # 3. 마지막 댓글 위치로 화면을 강제 이동 (이게 가장 안 튕깁니다)
#             driver.execute_script("arguments[0].scrollIntoView(true);", last_comment)
#         else:
#             # 댓글이 하나도 안 보일 경우 컨테이너 자체를 내림
#             driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_box)

#         # 4. 살짝 추가 스크롤을 주어 '더보기' 트리거를 발동시킴
#         driver.execute_script("document.querySelector('ytd-engagement-panel-section-list-renderer #content').scrollTop += 500;")
        
#         print(f"{a}번째 스크롤 및 데이터 로딩 중... (현재 로드된 댓글: {len(comments)}개)")
#         time.sleep(2.5) # 네트워크 로딩 속도를 고려해 여유 있게 대기

#     except Exception as e:
#         print(f"스크롤 시도 중 오류 발생: {e}")
#         break 

In [47]:
# 이게 되긴 되네??? 

from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import time

# ---------------------------------
# 댓글창 열린 뒤, 진짜 스크롤 요소 찾기
# ---------------------------------
driver.execute_script("""
const panel = document.querySelector('ytd-engagement-panel-section-list-renderer');
if (!panel) return;

const all = panel.querySelectorAll('*');
for (const el of all) {
    el.removeAttribute('data-scroll-target');
}

for (const el of all) {
    const style = window.getComputedStyle(el);
    const overflowY = style.overflowY;
    const isScrollable = (overflowY === 'auto' || overflowY === 'scroll');
    if (isScrollable && el.scrollHeight > el.clientHeight) {
        el.setAttribute('data-scroll-target', 'true');
        break;
    }
}
""")
time.sleep(5)

scroll_box = driver.find_element(By.CSS_SELECTOR, '[data-scroll-target="true"]')

print("scrollHeight:", driver.execute_script("return arguments[0].scrollHeight", scroll_box))
print("clientHeight:", driver.execute_script("return arguments[0].clientHeight", scroll_box))
print("scrollTop(before):", driver.execute_script("return arguments[0].scrollTop", scroll_box))

ActionChains(driver).move_to_element(scroll_box).click().perform()
time.sleep(5)

# ---------------------------------
# 원하는 댓글 수까지 로딩
# ---------------------------------
last_count = 0
no_change = 0
target_count = comment_cnt

while True:
    comments = driver.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer')
    current_count = len(comments)
    print(f"현재 로딩된 댓글 수 : {current_count}")

    if current_count >= target_count:
        print("원하는 댓글 수만큼 로드 완료")
        break

    # 1) 조금씩 scrollTop 내리기
    for _ in range(4):
        driver.execute_script("arguments[0].scrollTop += 250;", scroll_box)
        time.sleep(2)

    # 2) wheel 이벤트도 같이 보내기
    driver.execute_script("""
    arguments[0].dispatchEvent(new WheelEvent('wheel', {
        deltaY: 500,
        bubbles: true,
        cancelable: true
    }));
    """, scroll_box)

    time.sleep(3)

    new_count = len(driver.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer'))

    if new_count == current_count:
        no_change += 1
    else:
        no_change = 0

    if no_change >= 5:
        print("더 이상 새로운 댓글이 로딩되지 않습니다.")
        break

print("scrollTop(after):", driver.execute_script("return arguments[0].scrollTop", scroll_box))

scrollHeight: 8188
clientHeight: 803
scrollTop(before): 4787
현재 로딩된 댓글 수 : 60
원하는 댓글 수만큼 로드 완료
scrollTop(after): 4787


In [34]:
from selenium.webdriver.common.by import By
import re
import time

time.sleep(1)

# 큰 댓글 패널 찾기
anchored_panel = driver.find_element(By.CSS_SELECTOR, '#anchored-panel')
print("댓글 전체 패널 찾기 완료")

# 총 댓글 수 요소 찾기
comment_count_el = anchored_panel.find_element(By.CSS_SELECTOR, '#contextual-info')

# 원본 텍스트
comment_count_text = comment_count_el.text.strip()
print("원본 댓글 수 텍스트:", comment_count_text)

# 숫자만 추출
numbers = re.findall(r'\d+', comment_count_text)
if numbers:
    total_comment_count = int(numbers[0])
else:
    total_comment_count = 0

print("총 댓글 수:", total_comment_count)

댓글 전체 패널 찾기 완료
원본 댓글 수 텍스트: 280
총 댓글 수: 280


In [35]:
from selenium.webdriver.common.by import By
import time

time.sleep(2)

# 1. 댓글 패널 찾기
anchored_panel = driver.find_element(By.CSS_SELECTOR, '#anchored-panel')
print("댓글 패널 찾기 완료")

# 2. 댓글 목록 찾기
comment_items = anchored_panel.find_elements(By.CSS_SELECTOR, 'ytd-comment-thread-renderer')
print("현재 잡힌 댓글 수:", len(comment_items))

# 3. 첫 댓글 테스트
if len(comment_items) > 0:
    first_item = comment_items[0]

    try:
        author = first_item.find_element(By.CSS_SELECTOR, '#author-text').text
    except:
        author = ""

    try:
        content = first_item.find_element(By.CSS_SELECTOR, '#content-text').text
    except:
        content = ""

    print("작성자:", author)
    print("댓글내용:", content)
else:
    print("댓글이 없습니다.")

댓글 패널 찾기 완료
현재 잡힌 댓글 수: 100
작성자: @뇽야곽
댓글내용: 진짜 이런 솔직리뷰 너무 좋음..! 요즘 광고가 너무 많아서..


In [36]:
author_list = []
date_list = []
content_list = []
like_list = []

for idx, item in enumerate(comment_items, start=1):
    try:
        author = item.find_element(By.CSS_SELECTOR, '#author-text').text
    except:
        author = ""

    try:
        content = item.find_element(By.CSS_SELECTOR, '#content-text').text
    except:
        content = ""

    try:
        date_text = item.find_element(By.CSS_SELECTOR, '#published-time-text a').text
    except:
        try:
            date_text = item.find_element(By.CSS_SELECTOR, '#published-time-text').text
        except:
            date_text = ""

    try:
        like_count = item.find_element(By.CSS_SELECTOR, '#vote-count-middle').text
    except:
        like_count = ""

    author_list.append(author)
    date_list.append(date_text)
    content_list.append(content)
    like_list.append(like_count)

    print(f"{idx}번째 댓글 수집 완료")
    print("작성자:", author)
    print("작성일자:", date_text)
    print("댓글내용:", content)
    print("좋아요수:", like_count)
    print("-" * 50)

1번째 댓글 수집 완료
작성자: @뇽야곽
작성일자: 13일 전
댓글내용: 진짜 이런 솔직리뷰 너무 좋음..! 요즘 광고가 너무 많아서..
좋아요수: 2.8천
--------------------------------------------------
2번째 댓글 수집 완료
작성자: @letsgo_dwg
작성일자: 13일 전
댓글내용: 진짜 그로우어스 ㅋㅋㅋㅋㅋㅋ 사고 처음 쓰고 나갔는데 이곳저곳이 반짝거려서 뭔가했음... 손이 번쩍번쩍 옷이 번쩍번쩍;; 저 펄땡이 왜넣은건지
좋아요수: 4.7천
--------------------------------------------------
3번째 댓글 수집 완료
작성자: @Haaappiieee
작성일자: 13일 전
댓글내용: 그로우어스 진짜 인정 ㅠㅠㅠㅠㅠ 절대 사지 마세여 ㅠㅠㅠ 하필 두통이나 사서 돌아버리겠음 ㅠㅠ
좋아요수: 615
--------------------------------------------------
4번째 댓글 수집 완료
작성자: @체nana
작성일자: 13일 전
댓글내용: 와 그로우어스 좋다는 사람들을 많이봣는데 밤티템이엇구나..
유명한데 알고보면 밤티제품인게 ㅈㅉ많은듯
좋아요수: 1.2천
--------------------------------------------------
5번째 댓글 수집 완료
작성자: @meowtweew
작성일자: 13일 전
댓글내용: 그로우어스 노워시 트리트먼트는 마케팅으로 성공하긴 했지.. 저 똥템이 유명템이라는게 충격
좋아요수: 187
--------------------------------------------------
6번째 댓글 수집 완료
작성자: @nawo.20s
작성일자: 10일 전
댓글내용: 그로우어스는 머리 젖은 상태에서 두세번 뿌리고 바로 만지지 말고 머리 말리면 펄도 안나오고 ㄹㅇ 개쩔어요
좋아요수: 71
--------------------------------------------------
7번째 댓글 수집 완료
작성자: @정아빈-k

In [37]:
# ---------------------------------
# 실제 수집할 댓글 수 확정
# ---------------------------------
loaded_comment_count = len(comment_items)
actual_cnt = min(comment_cnt, loaded_comment_count)

print("\n" + "=" * 80)
print(f"현재 로딩된 댓글 수 : {loaded_comment_count}")
print(f"사용자가 요청한 댓글 수 : {comment_cnt}")
print(f"실제 수집할 댓글 수 : {actual_cnt}")
print("=" * 80)

# 원하는 개수만큼만 자르기
comment_items = comment_items[:actual_cnt]

# ---------------------------------
# 댓글 정보 수집
# ---------------------------------
author_list = []
date_list = []
content_list = []
like_list = []
video_no_list = []
review_no_list = []
video_url_list = []
video_title_list = []

for idx, item in enumerate(comment_items, start=1):
    try:
        author = item.find_element(By.CSS_SELECTOR, '#author-text').text.strip()
    except:
        author = ""

    try:
        content = item.find_element(By.CSS_SELECTOR, '#content-text').text.strip()
    except:
        content = ""

    try:
        date_text = item.find_element(By.CSS_SELECTOR, '#published-time-text a').text.strip()
    except:
        try:
            date_text = item.find_element(By.CSS_SELECTOR, '#published-time-text').text.strip()
        except:
            date_text = ""

    try:
        like_count = item.find_element(By.CSS_SELECTOR, '#vote-count-middle').text.strip()
    except:
        like_count = ""

    video_no_list.append(video_no)
    review_no_list.append(idx)
    video_url_list.append(video_url)
    video_title_list.append(video_title)
    author_list.append(author)
    date_list.append(date_text)
    content_list.append(content)
    like_list.append(like_count)

    print(f"{idx}번째 댓글 수집 완료")
    print("동영상번호:", video_no)
    print("리뷰번호:", idx)
    print("동영상URL:", video_url)
    print("동영상제목:", video_title)
    print("작성자:", author)
    print("작성일자:", date_text)
    print("댓글내용:", content)
    print("좋아요수:", like_count)
    print("-" * 50)

# ---------------------------------
# 데이터프레임 생성
# ---------------------------------
youtube_comment_df = pd.DataFrame({
    "동영상번호": video_no_list,
    "리뷰번호": review_no_list,
    "동영상URL": video_url_list,
    "동영상제목": video_title_list,
    "댓글작성자명": author_list,
    "댓글작성일자": date_list,
    "댓글내용": content_list,
    "좋아요수": like_list
})

print("\n데이터프레임 생성 완료")
print(youtube_comment_df.head())
print("총 수집 건수:", len(youtube_comment_df))


현재 로딩된 댓글 수 : 100
사용자가 요청한 댓글 수 : 100
실제 수집할 댓글 수 : 100
1번째 댓글 수집 완료
동영상번호: 1
리뷰번호: 1
동영상URL: https://www.youtube.com/shorts/EpRSKZ_S9us
동영상제목: 유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청
작성자: @뇽야곽
작성일자: 13일 전
댓글내용: 진짜 이런 솔직리뷰 너무 좋음..! 요즘 광고가 너무 많아서..
좋아요수: 2.8천
--------------------------------------------------
2번째 댓글 수집 완료
동영상번호: 1
리뷰번호: 2
동영상URL: https://www.youtube.com/shorts/EpRSKZ_S9us
동영상제목: 유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청
작성자: @letsgo_dwg
작성일자: 13일 전
댓글내용: 진짜 그로우어스 ㅋㅋㅋㅋㅋㅋ 사고 처음 쓰고 나갔는데 이곳저곳이 반짝거려서 뭔가했음... 손이 번쩍번쩍 옷이 번쩍번쩍;; 저 펄땡이 왜넣은건지
좋아요수: 4.7천
--------------------------------------------------
3번째 댓글 수집 완료
동영상번호: 1
리뷰번호: 3
동영상URL: https://www.youtube.com/shorts/EpRSKZ_S9us
동영상제목: 유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청
작성자: @Haaappiieee
작성일자: 13일 전
댓글내용: 그로우어스 진짜 인정 ㅠㅠㅠㅠㅠ 절대 사지 마세여 ㅠㅠㅠ 하필 두통이나 사서 돌아버리겠음 ㅠㅠ
좋아요수: 615
--------------------------------------------------
4번째 댓글 수집 완료
동영상번호: 1
리뷰번호: 4
동영상URL: https://www.youtube.com/shorts/EpRSKZ_S9us
동영상제목: 유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청
작성자: @

In [38]:
# ---------------------------------
# 저장 파일 경로와 이름 지정
# ---------------------------------
query_txt = search_keyword if collect_type == '2' else 'youtube_shorts'
sec_name = video_title if video_title != "" else 'shorts_comment'

# 파일명에 쓸 수 없는 문자 제거
sec_name = re.sub(r'[\\/:*?"<>|]', '', sec_name)

n = time.localtime()
s1 = '%04d-%02d-%02d-%02d-%02d-%02d' % (
    n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec
)

# 저장 폴더가 없으면 생성
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

# 최종 저장 폴더 생성
final_dir = os.path.join(save_folder, s1 + '-' + query_txt + '-' + sec_name)
os.makedirs(final_dir, exist_ok=True)

ff_dir = final_dir
ff_name = os.path.join(final_dir, s1 + '-' + query_txt + '-' + sec_name + '.txt')
fc_name = os.path.join(final_dir, s1 + '-' + query_txt + '-' + sec_name + '.csv')
fx_name = os.path.join(final_dir, s1 + '-' + query_txt + '-' + sec_name + '.xlsx')

# 이미지 저장용 폴더 (지금은 사용 안 해도 형식 유지)
img_dir = os.path.join(ff_dir, "images")
os.makedirs(img_dir, exist_ok=True)

s_time = time.time()

In [39]:
# ---------------------------------
# txt 저장
# ---------------------------------
with open(ff_name, 'w', encoding='utf-8') as f:
    f.write("=" * 100 + "\n")
    f.write("유튜브 숏츠 댓글 수집 결과\n")
    f.write("=" * 100 + "\n")
    f.write(f"동영상 제목 : {video_title}\n")
    f.write(f"동영상 URL : {video_url}\n")
    f.write(f"총 댓글 수 : {total_comment_count}\n")
    f.write(f"실제 수집 건수 : {len(youtube_comment_df)}\n")
    f.write("=" * 100 + "\n\n")

    for i in range(len(youtube_comment_df)):
        f.write(f"[{i+1}번째 댓글]\n")
        f.write("동영상번호: " + str(youtube_comment_df.loc[i, "동영상번호"]) + "\n")
        f.write("리뷰번호: " + str(youtube_comment_df.loc[i, "리뷰번호"]) + "\n")
        f.write("동영상URL: " + str(youtube_comment_df.loc[i, "동영상URL"]) + "\n")
        f.write("동영상제목: " + str(youtube_comment_df.loc[i, "동영상제목"]) + "\n")
        f.write("댓글작성자명: " + str(youtube_comment_df.loc[i, "댓글작성자명"]) + "\n")
        f.write("댓글작성일자: " + str(youtube_comment_df.loc[i, "댓글작성일자"]) + "\n")
        f.write("댓글내용: " + str(youtube_comment_df.loc[i, "댓글내용"]) + "\n")
        f.write("좋아요수: " + str(youtube_comment_df.loc[i, "좋아요수"]) + "\n")
        f.write("-" * 100 + "\n")

# ---------------------------------
# csv, xlsx 저장
# ---------------------------------
youtube_comment_df.to_csv(fc_name, encoding="utf-8-sig", index=False)
youtube_comment_df.to_excel(fx_name, index=False, engine='openpyxl')

e_time = time.time()
t_time = e_time - s_time

print("\n")
print("=" * 100)
print("1. 요청된 총 %s 건 중에서 실제 크롤링 된 건수는 %s 건입니다" % (comment_cnt, len(youtube_comment_df)))
print("2. 총 소요시간은 %s 초 입니다" % round(t_time, 1))
print("3. txt 파일명 : %s" % ff_name)
print("4. csv 파일명 : %s" % fc_name)
print("5. xlsx 파일명 : %s" % fx_name)
print("=" * 100)



1. 요청된 총 100 건 중에서 실제 크롤링 된 건수는 100 건입니다
2. 총 소요시간은 2.0 초 입니다
3. txt 파일명 : c:\py_temp\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청.txt
4. csv 파일명 : c:\py_temp\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청.csv
5. xlsx 파일명 : c:\py_temp\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청\2026-03-12-18-05-55-youtube_shorts-유명한데 별로인.. 올영 비추템👎🏽 올영세일 전 필수 시청.xlsx
